In [ ]:
import torch
from torch import nn
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType


class SingleTokenMLP(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(paragraph_dim, 2048),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(2048, decoder_hidden_dim)
        )

    def forward(self, x):
        return self.net(x)


class EndToEndInverter(nn.Module):
    def __init__(self, paragraph_dim, decoder_hidden_dim, prefix_len, decoder_model):
        super().__init__()

        self.prefix_len = prefix_len
        self.decoder_hidden_dim = decoder_hidden_dim

        self.m_parallel_mlps = nn.ModuleList([
            SingleTokenMLP(paragraph_dim, decoder_hidden_dim)
            for _ in range(prefix_len)
        ])

        self.decoder = decoder_model

    def forward(self, paragraph_embs, text_input_ids, attention_mask):
        device = next(self.parameters()).device

        paragraph_embs = paragraph_embs.to(device)
        text_input_ids = text_input_ids.to(device)
        attention_mask = attention_mask.to(device)

        batch_size = paragraph_embs.shape[0]

        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)

        text_embeds = self.decoder.get_input_embeddings()(text_input_ids)

        prefix_embeds = prefix_embeds.to(dtype=text_embeds.dtype)
        inputs_embeds = torch.cat([prefix_embeds, text_embeds], dim=1)

        prefix_labels = torch.full(
            (batch_size, self.prefix_len),
            -100,
            dtype=torch.long,
            device=device
        )

        labels = torch.cat([prefix_labels, text_input_ids], dim=1)

        prefix_mask = torch.ones(
            (batch_size, self.prefix_len),
            dtype=attention_mask.dtype,
            device=device
        )

        concat_mask = torch.cat([prefix_mask, attention_mask], dim=1)

        outputs = self.decoder(
            inputs_embeds=inputs_embeds,
            attention_mask=concat_mask,
            labels=labels
        )

        return outputs.loss

    def generate(self, paragraph_embs, tokenizer, max_new_tokens=128):
        device = next(self.parameters()).device
        paragraph_embs = paragraph_embs.to(device)

        outputs = [mlp(paragraph_embs) for mlp in self.m_parallel_mlps]
        prefix_embeds = torch.stack(outputs, dim=1)

        prefix_embeds = prefix_embeds.to(device=device, dtype=self.decoder.dtype)

        outputs = self.decoder.generate(
            inputs_embeds=prefix_embeds,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,
            repetition_penalty=1.2
        )

        return tokenizer.batch_decode(outputs, skip_special_tokens=True)


MODEL_NAME = "Qwen/Qwen3-0.6B"
PARAGRAPH_DIM = 1024
PREFIX_LEN = 32

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


print(f"Loading base decoder {MODEL_NAME}...")

base_decoder = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map={"": 0}  
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

decoder_with_lora = get_peft_model(base_decoder, lora_config)

decoder_hidden_size = base_decoder.config.hidden_size


model = EndToEndInverter(
    paragraph_dim=PARAGRAPH_DIM,
    decoder_hidden_dim=decoder_hidden_size,
    prefix_len=PREFIX_LEN,
    decoder_model=decoder_with_lora
)

model = model.to(device)
model.train()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\nModel initialized!")
print(f"Decoder hidden size: {decoder_hidden_size}")
print(f"Trainable params: {trainable_params:,}")

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download


facts = [
    "Paris is the capital of France and it is known for its art museums historic architecture and famous landmarks like the Eiffel Tower and the Louvre museum",
    "The Earth revolves around the Sun once every year and this motion along with its tilted axis is responsible for the changing seasons we experience",
    "Water boils at one hundred degrees Celsius under standard atmospheric pressure and this property is commonly used in cooking scientific experiments and industrial processes",
    "The human brain controls the entire body by sending electrical signals through the nervous system allowing us to think move feel emotions and respond to our environment",
    "Mount Everest is the highest mountain above sea level and it is located in the Himalayas on the border between Nepal and China attracting climbers from around the world"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen3-Embedding-0.6B"

encoder_model = AutoModel.from_pretrained(MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

encoder_model.to(device)
encoder_model.eval()

inputs = tokenizer(
    facts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    outputs = encoder_model(**inputs)

token_embeddings = outputs.last_hidden_state
attention_mask = inputs["attention_mask"]

mask_expanded = attention_mask.unsqueeze(-1).float()
masked = token_embeddings * mask_expanded

sentence_embeddings = masked.sum(dim=1) / mask_expanded.sum(dim=1)
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-0.6B",
    device_map={"": 0}, 
    trust_remote_code=True
)

paragraph_dim = sentence_embeddings.shape[1]

model = EndToEndInverter(
    paragraph_dim=paragraph_dim,
    decoder_hidden_dim=base_model.config.hidden_size,
    prefix_len=32,
    decoder_model=base_model
)

file_path = hf_hub_download(
    repo_id="Subhav-K/qwen-embedding-inverter-v1",
    filename="entire_end_to_end_aux_model.pt"
)

state_dict = torch.load(file_path, map_location="cpu")
model.load_state_dict(state_dict, strict=False)

model = model.to(device)
model.eval()

sentence_embeddings = sentence_embeddings.to(device)

outputs = model.generate(
    sentence_embeddings,
    tokenizer,
    max_new_tokens=40
)

for i, out in enumerate(outputs):
    print(f"\n--- Sample 1 {i+1} ---")
    print("Original : ",facts[i])
    print("Generated Output : ",out)

In [ ]:
!nvidia-smi

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from huggingface_hub import hf_hub_download

device = "cuda:0"

def load_model(state_dict_repo, filename):
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-0.6B",
        device_map={"": 0},
        trust_remote_code=True
    )

    model = EndToEndInverter(
        paragraph_dim=sentence_embeddings.shape[1],
        decoder_hidden_dim=base_model.config.hidden_size,
        prefix_len=32,
        decoder_model=base_model
    )

    file_path = hf_hub_download(
        repo_id=state_dict_repo,
        filename=filename
    )

    state_dict = torch.load(file_path, map_location="cpu")
    model.load_state_dict(state_dict, strict=False)

    model = model.to(device)
    model.eval()
    return model


sentence_embeddings = sentence_embeddings.to(device)
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F
import torch

sim_model = SentenceTransformer("all-MiniLM-L6-v2").to(device)

model_A = load_model("Subhav-K/qwen-embedding-inverter-v1", "entire_end_to_end_aux_model.pt")
outputs_A = model_A.generate(sentence_embeddings, tokenizer, max_new_tokens=40)

del model_A
torch.cuda.empty_cache()

model_B = load_model("jg-eno/ReLoDer", "best_end_to_end_model.pt")
outputs_B = model_B.generate(sentence_embeddings, tokenizer, max_new_tokens=40)

for i in range(len(facts)):
    orig_emb = sim_model.encode(facts[i], convert_to_tensor=True).to(device)
    
    outA_emb = sim_model.encode(outputs_A[i], convert_to_tensor=True).to(device)
    outB_emb = sim_model.encode(outputs_B[i], convert_to_tensor=True).to(device)

    sim_A = F.cosine_similarity(orig_emb, outA_emb, dim=0).item()
    sim_B = F.cosine_similarity(orig_emb, outB_emb, dim=0).item()

    print(f"\n=== Sample {i+1} ===")
    print("Input:", facts[i])
    
    print("\nSubhav-K/qwen-embedding-inverter-v1 Output:", outputs_A[i])
    print("Cosine Sim A:", round(sim_A, 4))
    
    print("\njg-eno/ReLoDer Output:", outputs_B[i])
    print("Cosine Sim B:", round(sim_B, 4))